# ML Teaching Essentials -- Model Diagnostics
## TL;DR -- Plain English Introduction

**What separates good ML engineers from great ones?** Knowing WHY a model fails and HOW to fix it. This notebook teaches the diagnostic skills that interviewers test.

**The 5 most common ML failures (and fixes):**
1. **High bias (underfitting)** -- model too simple -> use more complex model or add features
2. **High variance (overfitting)** -- model memorizes training data -> add regularization, more data, dropout
3. **Vanishing gradients** -- gradients near 0 in deep networks -> use ReLU, batch norm, residual connections
4. **Exploding gradients** -- gradients blow up -> gradient clipping (`clip_grad_norm_`)
5. **Data leakage** -- test data contaminated training -> strict train/val/test splits before ANY processing

**No ML background?**
- Training loss = how wrong is my model on data it has seen?
- Validation loss = how wrong on data it has NOT seen?
- If train loss << val loss: overfitting (memorizing, not learning)
- If both losses are high: underfitting (not learning at all)

**Learning path:** 00/02 (ML fundamentals) -> 00/06 (PyTorch) -> This notebook -> 05/01 (Deep learning) -> 10/01 (Fine-tuning)

## Learning Objectives

- [ ] Draw and interpret the bias-variance tradeoff with polynomial regression
- [ ] Generate and read learning curves to diagnose overfitting vs underfitting
- [ ] Generate and read validation curves to select hyperparameters
- [ ] Evaluate model calibration using reliability diagrams
- [ ] Apply Platt scaling and temperature scaling to improve calibration
- [ ] Follow a systematic diagnostic checklist when a model underperforms
- [ ] Debug buggy ML code using diagnostic reasoning
- [ ] Answer 5 common interview questions on model diagnostics

## Concepts for Beginners (Glossary)

| Term | Plain English |
|------|---------------|
| **bias** | Systematic error -- a biased model makes the same type of mistake consistently |
| **variance** | Sensitivity to training data -- high-variance model changes dramatically with different training sets |
| **bias-variance tradeoff** | Simple models have high bias (underfit); complex models have high variance (overfit) |
| **underfitting** | Model is too simple to capture the pattern -- both training and validation error are high |
| **overfitting** | Model memorises training data -- low training error but high validation error |
| **learning curve** | Plot of train/val error vs training set size -- reveals whether you need more data or less complexity |
| **validation curve** | Plot of train/val error vs hyperparameter -- shows the underfitting->overfitting transition |
| **calibration** | A model is calibrated if its confidence scores match actual frequencies (70% prediction -> 70% correct) |
| **reliability diagram** | Plot of mean predicted probability vs actual fraction of positives -- checks calibration |
| **Platt scaling** | Fits a logistic regression on top of raw model outputs to improve calibration |
| **temperature scaling** | Divides logits by a learnable temperature T before softmax -- simplest calibration method |
| **early stopping** | Stop training when validation loss stops improving -- a form of implicit regularisation |

## Prerequisites & Cross-References

| | |
|--|--|
| Prerequisites | 00/02 (NumPy/matplotlib), 00/03-05 (ML fundamentals) |
| You Are Here | Module 09/01 -- Model Diagnostics |
| Next Steps | Any module requiring model training -- this is a standalone reference |
| Goal | Diagnose underfitting/overfitting from curves; evaluate calibration; apply systematic debugging |

### Cross-References
- **Builds on:** 00/02 (plotting), 04/01 (ML fundamentals -- bias-variance introduced there)
- **Used by:** ALL modules with trained models -- apply these diagnostics to every notebook's training cell
- **Standalone:** This notebook can be read independently as a reference guide

### From Scratch? Start Here:
1. [StatQuest bias-variance video](https://www.youtube.com/watch?v=EuBBz3bI-aA) -- 10 min, best visual explanation
2. [sklearn learning_curve docs](https://scikit-learn.org/stable/modules/learning_curve.html)
3. 00/02 (this repo) -- NumPy/matplotlib for plotting diagnostics
4. This notebook -- model diagnostics

**Time:** 4-6h | **Difficulty:** Intermediate

---
# Section 1: Bias-Variance Tradeoff

The **bias-variance tradeoff** is the central tension in machine learning:

$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Noise}$$

- **Bias** = error from wrong assumptions (model too simple)
- **Variance** = error from sensitivity to training data (model too complex)
- **Irreducible noise** = randomness in the data itself

We will visualize this with polynomial regression on a sine curve.

In [ ]:
# Cell 1: Imports and setup (all cells use numpy + sklearn + matplotlib only)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import learning_curve, validation_curve
from sklearn.svm import SVC
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import brier_score_loss
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("All imports successful!")
print("Module 09: Model Diagnostics -- bias-variance, learning curves, calibration")

> **Expected output:** `All imports successful!` and module description. If any import fails, run `pip install scikit-learn matplotlib numpy`.

In [ ]:
# Cell 2: Bias-Variance Tradeoff Visualization
# Fit polynomials of degree 1 (underfit), 4 (good), 15 (overfit) to noisy sine data

np.random.seed(42)
X = np.sort(np.random.uniform(0, 1, 30)).reshape(-1, 1)
y = np.sin(2 * np.pi * X).ravel() + np.random.normal(0, 0.3, 30)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
X_plot = np.linspace(0, 1, 100).reshape(-1, 1)
for ax, degree, title in zip(axes, [1, 4, 15],
        ['Underfitting (degree=1)', 'Good fit (degree=4)', 'Overfitting (degree=15)']):
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X, y)
    ax.scatter(X, y, s=20, color='steelblue', label='Data')
    ax.plot(X_plot, model.predict(X_plot), 'r-', lw=2, label=f'Degree {degree}')
    ax.plot(X_plot, np.sin(2 * np.pi * X_plot), 'g--', alpha=0.5, label='True function')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_ylim(-2, 2)
plt.suptitle('Bias-Variance Tradeoff Visualized', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

> **Expected output:** Three side-by-side plots:
> - **Left (degree=1):** A straight line that misses the sine curve -- high bias (underfitting)
> - **Middle (degree=4):** A smooth curve that follows the sine pattern -- good fit
> - **Right (degree=15):** A wiggly curve that passes through every point -- high variance (overfitting)
>
> The green dashed line shows the true function. The red line is the model's prediction.

In [ ]:
# Cell 3: Quantify the bias-variance tradeoff
# Train multiple models on bootstrap samples to measure bias and variance

np.random.seed(42)
n_bootstraps = 200
X_test = np.linspace(0, 1, 100).reshape(-1, 1)
y_true = np.sin(2 * np.pi * X_test).ravel()

degrees = range(1, 16)
bias_sq_list, variance_list, mse_list = [], [], []

for degree in degrees:
    predictions = np.zeros((n_bootstraps, len(X_test)))
    for i in range(n_bootstraps):
        X_boot = np.sort(np.random.uniform(0, 1, 30)).reshape(-1, 1)
        y_boot = np.sin(2 * np.pi * X_boot).ravel() + np.random.normal(0, 0.3, 30)
        model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
        model.fit(X_boot, y_boot)
        predictions[i] = model.predict(X_test)
    
    mean_pred = predictions.mean(axis=0)
    bias_sq = np.mean((mean_pred - y_true) ** 2)
    variance = np.mean(predictions.var(axis=0))
    mse = np.mean(np.mean((predictions - y_true) ** 2, axis=1))
    
    bias_sq_list.append(bias_sq)
    variance_list.append(variance)
    mse_list.append(mse)

plt.figure(figsize=(8, 5))
plt.plot(list(degrees), bias_sq_list, 'b-o', label='Bias$^2$', markersize=4)
plt.plot(list(degrees), variance_list, 'r-o', label='Variance', markersize=4)
plt.plot(list(degrees), mse_list, 'k--o', label='Total MSE', markersize=4)
plt.xlabel('Polynomial Degree (Model Complexity)', fontsize=12)
plt.ylabel('Error', fontsize=12)
plt.title('Bias-Variance Decomposition', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_degree = list(degrees)[np.argmin(mse_list)]
print(f"Best degree (lowest total MSE): {best_degree}")
print(f"At degree {best_degree}: Bias^2={bias_sq_list[best_degree-1]:.4f}, "
      f"Variance={variance_list[best_degree-1]:.4f}, MSE={mse_list[best_degree-1]:.4f}")

> **Expected output:** A plot showing bias squared (blue, decreasing), variance (red, increasing), and total MSE (black dashed, U-shaped). The optimal complexity is where the black curve is lowest (around degree 3-5). This is the bias-variance tradeoff in action.

> **Common Mistake:** Students often think "more complex = better" because training error always decreases with complexity. But the plot above shows that total error (which includes variance) goes UP after the sweet spot. Always evaluate on held-out data, never on training data alone.

---
# Section 2: Learning Curves

A **learning curve** plots training and validation performance as a function of **training set size**.

It answers the question: **"Will getting more data help?"**

| Pattern | Diagnosis | Action |
|---------|-----------|--------|
| Both curves high, converged | High bias (underfitting) | More complex model, more features |
| Train low, val high, gap persists | High variance (overfitting) | More data, regularization |
| Both curves low, converged | Good fit | Ship it |

In [ ]:
# Cell 4: Learning Curves -- High Bias vs High Variance

np.random.seed(42)
X_lc = np.random.randn(300, 2)
y_lc = (X_lc[:, 0] + X_lc[:, 1] > 0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, model, name in zip(axes,
        [SVC(kernel='linear'), SVC(kernel='rbf', C=100)],
        ['Linear SVM (High Bias)', 'RBF SVM C=100 (High Variance)']):
    sizes, train_scores, val_scores = learning_curve(
        model, X_lc, y_lc,
        train_sizes=np.linspace(0.1, 1.0, 10), cv=5, scoring='accuracy')
    ax.plot(sizes, train_scores.mean(1), 'b-o', label='Training')
    ax.plot(sizes, val_scores.mean(1), 'r-o', label='Validation')
    ax.fill_between(sizes,
        train_scores.mean(1) - train_scores.std(1),
        train_scores.mean(1) + train_scores.std(1), alpha=0.1, color='blue')
    ax.fill_between(sizes,
        val_scores.mean(1) - val_scores.std(1),
        val_scores.mean(1) + val_scores.std(1), alpha=0.1, color='red')
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('Accuracy')
    ax.set_title(name)
    ax.legend()
    ax.grid(alpha=0.3)
plt.suptitle('Learning Curves: Diagnosing Bias vs Variance', fontweight='bold')
plt.tight_layout()
plt.show()

> **Expected output:** Two learning curve plots:
> - **Left (Linear SVM):** Training and validation scores converge together at a moderate level with a small gap. This is characteristic of **high bias** -- more data will not help much.
> - **Right (RBF SVM C=100):** Training score is near perfect but validation score is lower with a persistent gap. This is **high variance** -- more data or regularization would help.
>
> The shaded areas show +/- 1 standard deviation across CV folds.

In [ ]:
# Cell 5: Learning Curve Helper Function (reusable)

def plot_learning_curve(estimator, X, y, title='Learning Curve',
                        train_sizes=np.linspace(0.1, 1.0, 10), cv=5, ax=None):
    """Plot a learning curve with confidence bands.
    
    Parameters
    ----------
    estimator : sklearn estimator
    X : array-like, features
    y : array-like, labels
    title : str
    train_sizes : array-like, fractions of training data to use
    cv : int, number of CV folds
    ax : matplotlib axes (optional)
    
    Returns
    -------
    dict with keys: 'train_sizes', 'train_mean', 'val_mean', 'gap'
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4))
    
    sizes, train_scores, val_scores = learning_curve(
        estimator, X, y, train_sizes=train_sizes, cv=cv, scoring='accuracy')
    
    train_mean = train_scores.mean(axis=1)
    val_mean = val_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_std = val_scores.std(axis=1)
    
    ax.plot(sizes, train_mean, 'b-o', label='Training', markersize=4)
    ax.plot(sizes, val_mean, 'r-o', label='Validation', markersize=4)
    ax.fill_between(sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    ax.fill_between(sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='red')
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('Accuracy')
    ax.set_title(title, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)
    
    gap = train_mean[-1] - val_mean[-1]
    print(f"{title}:")
    print(f"  Final train accuracy: {train_mean[-1]:.3f}")
    print(f"  Final val accuracy:   {val_mean[-1]:.3f}")
    print(f"  Gap (train - val):    {gap:.3f}")
    if gap > 0.1:
        print("  Diagnosis: HIGH VARIANCE (overfitting) -- try more data or regularization")
    elif val_mean[-1] < 0.8:
        print("  Diagnosis: HIGH BIAS (underfitting) -- try more complex model")
    else:
        print("  Diagnosis: Good fit")
    
    return {'train_sizes': sizes, 'train_mean': train_mean,
            'val_mean': val_mean, 'gap': gap}

# Demo
result = plot_learning_curve(SVC(kernel='rbf', C=1.0), X_lc, y_lc,
                             title='RBF SVM (C=1.0)')
plt.tight_layout()
plt.show()

> **Expected output:** A learning curve plot for RBF SVM with C=1.0, plus printed diagnostics showing final train/val accuracy, the gap, and a diagnosis string.

---
# Section 3: Validation Curves

A **validation curve** plots training and validation performance as a function of a **hyperparameter**.

It answers the question: **"What is the best value for this hyperparameter?"**

- Left side of the curve: hyperparameter causes underfitting
- Right side of the curve: hyperparameter causes overfitting
- Sweet spot: where validation score is highest

In [ ]:
# Cell 6: Validation Curve -- SVM C parameter

np.random.seed(42)
X_vc = np.random.randn(200, 2)
y_vc = (X_vc[:, 0] ** 2 + X_vc[:, 1] ** 2 > 1).astype(int)

C_range = np.logspace(-3, 3, 15)

train_scores, val_scores = validation_curve(
    SVC(kernel='rbf'), X_vc, y_vc,
    param_name='C', param_range=C_range,
    cv=5, scoring='accuracy')

plt.figure(figsize=(8, 5))
plt.semilogx(C_range, train_scores.mean(1), 'b-o', label='Training', markersize=4)
plt.semilogx(C_range, val_scores.mean(1), 'r-o', label='Validation', markersize=4)
plt.fill_between(C_range,
    train_scores.mean(1) - train_scores.std(1),
    train_scores.mean(1) + train_scores.std(1), alpha=0.1, color='blue')
plt.fill_between(C_range,
    val_scores.mean(1) - val_scores.std(1),
    val_scores.mean(1) + val_scores.std(1), alpha=0.1, color='red')
plt.xlabel('C (Regularization Parameter)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Validation Curve: SVM C Parameter', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_idx = np.argmax(val_scores.mean(1))
print(f"Best C: {C_range[best_idx]:.4f}")
print(f"Best validation accuracy: {val_scores.mean(1)[best_idx]:.3f}")

> **Expected output:** A validation curve plot with log-scale x-axis (C values from 0.001 to 1000). Training score increases monotonically toward 1.0. Validation score rises, peaks at an intermediate C value, then may plateau or slightly drop. The best C value is printed below.
>
> Key insight: small C = heavy regularization (underfitting), large C = little regularization (overfitting).

In [ ]:
# Cell 7: Validation Curve -- Polynomial Degree (Ridge Regression)

np.random.seed(42)
X_vr = np.sort(np.random.uniform(0, 1, 80)).reshape(-1, 1)
y_vr = np.sin(2 * np.pi * X_vr).ravel() + np.random.normal(0, 0.3, 80)

degrees = list(range(1, 16))
train_mses, val_mses = [], []

from sklearn.model_selection import cross_val_score

for d in degrees:
    model = make_pipeline(PolynomialFeatures(d), Ridge(alpha=0.01))
    # negative MSE (sklearn convention: higher is better)
    scores = cross_val_score(model, X_vr, y_vr, cv=5, scoring='neg_mean_squared_error')
    val_mses.append(-scores.mean())
    model.fit(X_vr, y_vr)
    train_mses.append(np.mean((model.predict(X_vr) - y_vr) ** 2))

plt.figure(figsize=(8, 5))
plt.plot(degrees, train_mses, 'b-o', label='Training MSE', markersize=5)
plt.plot(degrees, val_mses, 'r-o', label='Validation MSE', markersize=5)
plt.xlabel('Polynomial Degree', fontsize=12)
plt.ylabel('Mean Squared Error', fontsize=12)
plt.title('Validation Curve: Polynomial Degree', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_d = degrees[np.argmin(val_mses)]
print(f"Best polynomial degree: {best_d}")
print(f"Training MSE at best degree: {train_mses[best_d-1]:.4f}")
print(f"Validation MSE at best degree: {val_mses[best_d-1]:.4f}")

> **Expected output:** Training MSE decreases steadily with degree. Validation MSE decreases, reaches a minimum around degree 3-5, then increases. The gap between curves widens at high degrees -- classic overfitting signature.

---
## CHECKPOINT: Sections 1-3 Complete

**Pause here and verify you understand:**
1. Can you sketch the bias-variance tradeoff curve from memory?
2. Given a learning curve, can you diagnose high bias vs high variance?
3. Given a validation curve, can you pick the best hyperparameter value?

If yes, continue. If not, re-read the sections above.

---

---
# Section 4: Model Calibration

**Calibration** answers: "When my model says 70% probability, is it actually correct 70% of the time?"

This matters enormously in **medical and biological applications** where predicted probabilities drive clinical decisions.

**Key concepts:**
- A **reliability diagram** plots predicted probability bins vs actual fraction of positives
- A perfectly calibrated model lies on the diagonal
- **Platt scaling:** fit a logistic regression on model outputs to correct calibration
- **Temperature scaling:** divide logits by temperature T before softmax (neural nets)
- **Expected Calibration Error (ECE):** weighted average of |predicted - actual| across bins

In [ ]:
# Cell 8: Reliability Diagram (Calibration Plot)

np.random.seed(42)
X_cal = np.random.randn(1000, 5)
y_cal = (X_cal @ np.array([1, 0.5, -0.5, 0.3, -0.2]) > 0).astype(int)

# Split into train/test
X_train_cal, X_test_cal = X_cal[:800], X_cal[800:]
y_train_cal, y_test_cal = y_cal[:800], y_cal[800:]

model_cal = LogisticRegression().fit(X_train_cal, y_train_cal)
probs = model_cal.predict_proba(X_test_cal)[:, 1]
fraction_pos, mean_predicted = calibration_curve(y_test_cal, probs, n_bins=10)

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')
plt.plot(mean_predicted, fraction_pos, 'rs-', label='Logistic Regression')
plt.xlabel('Mean Predicted Probability', fontsize=12)
plt.ylabel('Fraction of Positives', fontsize=12)
plt.title('Reliability Diagram (Calibration Plot)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

brier = brier_score_loss(y_test_cal, probs)
print(f"Brier Score: {brier:.4f} (lower is better, 0 = perfect)")

> **Expected output:** A reliability diagram with the diagonal (dashed black) as the reference. The red squares (Logistic Regression) should lie close to the diagonal, since logistic regression is generally well-calibrated. Brier score should be small (< 0.15).
>
> Note: Logistic regression is one of the few models that is well-calibrated by default. SVMs, random forests, and neural networks are typically NOT well-calibrated.

## 🏁 Checkpoint — Pause and Verify

Before continuing, make sure you can:
1. Look at a learning curve and immediately identify: high bias or high variance?
2. Explain the bias-variance tradeoff in one sentence without looking at notes
3. Produce a calibration plot for any classifier using `calibration_curve`

**If any of these feel shaky**, re-read the sections above before continuing to the exercises.

In [ ]:
# Cell 9: Comparing Calibration Across Models

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

models = {
    'Logistic Regression': LogisticRegression(),
    'SVM (Platt scaling)': SVC(probability=True),
    'Random Forest': RandomForestClassifier(n_estimators=50, random_state=42),
    'Naive Bayes': GaussianNB(),
}

plt.figure(figsize=(7, 7))
plt.plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')
colors = ['red', 'blue', 'green', 'orange']

for (name, model), color in zip(models.items(), colors):
    model.fit(X_train_cal, y_train_cal)
    probs = model.predict_proba(X_test_cal)[:, 1]
    fraction_pos, mean_predicted = calibration_curve(y_test_cal, probs, n_bins=10)
    brier = brier_score_loss(y_test_cal, probs)
    plt.plot(mean_predicted, fraction_pos, 's-', color=color,
             label=f'{name} (Brier={brier:.3f})')

plt.xlabel('Mean Predicted Probability', fontsize=12)
plt.ylabel('Fraction of Positives', fontsize=12)
plt.title('Calibration Comparison Across Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

> **Expected output:** Four calibration curves plotted together. Logistic regression typically tracks the diagonal best. Random Forest tends to push probabilities toward 0.5 (sigmoid-shaped deviation). Naive Bayes tends to push predictions toward 0 and 1 (overconfident). SVM with Platt scaling is usually reasonable.

In [ ]:
# Cell 10: Platt Scaling (Post-hoc Calibration)
# CalibratedClassifierCV applies Platt scaling (sigmoid) or isotonic regression

from sklearn.ensemble import RandomForestClassifier

np.random.seed(42)
rf_uncalibrated = RandomForestClassifier(n_estimators=50, random_state=42)
rf_calibrated = CalibratedClassifierCV(rf_uncalibrated, method='sigmoid', cv=5)

# Fit both
rf_uncalibrated.fit(X_train_cal, y_train_cal)
rf_calibrated.fit(X_train_cal, y_train_cal)

# Get probabilities
probs_uncal = rf_uncalibrated.predict_proba(X_test_cal)[:, 1]
probs_cal = rf_calibrated.predict_proba(X_test_cal)[:, 1]

plt.figure(figsize=(7, 7))
plt.plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')

for probs, name, color in [(probs_uncal, 'RF (uncalibrated)', 'red'),
                            (probs_cal, 'RF + Platt scaling', 'blue')]:
    frac, mean_pred = calibration_curve(y_test_cal, probs, n_bins=10)
    brier = brier_score_loss(y_test_cal, probs)
    plt.plot(mean_pred, frac, 's-', color=color, label=f'{name} (Brier={brier:.3f})')

plt.xlabel('Mean Predicted Probability', fontsize=12)
plt.ylabel('Fraction of Positives', fontsize=12)
plt.title('Effect of Platt Scaling on Random Forest', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Brier (uncalibrated): {brier_score_loss(y_test_cal, probs_uncal):.4f}")
print(f"Brier (Platt scaling): {brier_score_loss(y_test_cal, probs_cal):.4f}")

> **Expected output:** Two calibration curves for Random Forest: one before (red, deviates from diagonal) and one after Platt scaling (blue, closer to diagonal). The Brier score should decrease after calibration.

In [ ]:
# Cell 11: Temperature Scaling (for Neural Networks)
# Simulated: temperature scaling divides logits by T before softmax

def softmax(logits):
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)

def temperature_scale(logits, T):
    """Apply temperature scaling to logits."""
    return softmax(logits / T)

# Simulate overconfident neural network logits
np.random.seed(42)
n_samples = 500
true_probs = np.random.uniform(0.3, 0.7, n_samples)
y_temp = (np.random.random(n_samples) < true_probs).astype(int)

# Overconfident logits (high magnitude)
logits = np.column_stack([
    -3 * (true_probs - 0.5),  # class 0 logit
    3 * (true_probs - 0.5)    # class 1 logit (overconfident by factor 3)
])

temperatures = [0.5, 1.0, 1.5, 2.0, 3.0]
plt.figure(figsize=(8, 6))
plt.plot([0, 1], [0, 1], 'k--', label='Perfect')

for T in temperatures:
    scaled_probs = temperature_scale(logits, T)[:, 1]
    frac, mean_pred = calibration_curve(y_temp, scaled_probs, n_bins=8)
    plt.plot(mean_pred, frac, 'o-', label=f'T={T}', markersize=5)

plt.xlabel('Mean Predicted Probability', fontsize=12)
plt.ylabel('Fraction of Positives', fontsize=12)
plt.title('Temperature Scaling: Effect on Calibration', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Temperature scaling intuition:")
print("  T < 1: Makes model MORE confident (sharper softmax)")
print("  T = 1: No change (original softmax)")
print("  T > 1: Makes model LESS confident (softer softmax)")
print("  Optimal T is learned on a held-out calibration set")

> **Expected output:** Multiple calibration curves at different temperatures. T=1 (original) is overconfident (S-shaped deviation). Higher T values (1.5-2.0) should bring the curve closer to the diagonal. T=0.5 makes it worse (even more overconfident).

---
# Section 5: Diagnostic Checklist

## Diagnostic Checklist -- When Your Model Underperforms

| Symptom | Diagnosis | Fix |
|---------|-----------|-----|
| Train error high, val error high | **High bias** (underfitting) | Increase model complexity, add features, reduce regularisation |
| Train error low, val error high | **High variance** (overfitting) | More data, regularisation, dropout, early stopping |
| Train error low, val error low but test error high | **Data leakage** | Check for target leakage in features, fix train/test split |
| Good accuracy but wrong confident predictions | **Poor calibration** | Apply Platt scaling or temperature scaling |
| Performance varies across CV folds | **Unstable model** | Ensemble methods, more data, simpler model |
| Performance drops over time | **Data drift** | Monitor input distribution, retrain periodically |

### Step-by-Step Diagnostic Workflow

1. **Check data quality** -- missing values? class imbalance? label noise?
2. **Plot learning curves** -- high bias or high variance?
3. **Plot validation curves** -- is the hyperparameter in the right range?
4. **Check calibration** -- are probability estimates reliable?
5. **Check feature importance** -- are the right features being used?
6. **Check for leakage** -- is future information leaking into training?
7. **Check residuals** -- are errors random or systematic?

**Rule of thumb:** Change ONE thing at a time. Log everything.

In [ ]:
# Cell 12: Automated Diagnostic Report
# Given train/val scores, produce a diagnosis

def diagnose_model(train_score, val_score, threshold_high_bias=0.75,
                   threshold_gap=0.1):
    """Produce a diagnostic report given train and validation scores.
    
    Parameters
    ----------
    train_score : float, training accuracy (0-1)
    val_score : float, validation accuracy (0-1)
    threshold_high_bias : float, below this = underfitting
    threshold_gap : float, gap above this = overfitting
    
    Returns
    -------
    str : diagnosis and recommended actions
    """
    gap = train_score - val_score
    
    print("=" * 50)
    print("MODEL DIAGNOSTIC REPORT")
    print("=" * 50)
    print(f"Training score:   {train_score:.3f}")
    print(f"Validation score: {val_score:.3f}")
    print(f"Gap:              {gap:.3f}")
    print("-" * 50)
    
    if train_score < threshold_high_bias and val_score < threshold_high_bias:
        diagnosis = "HIGH BIAS (Underfitting)"
        actions = [
            "Increase model complexity (more layers, higher degree)",
            "Add more features or feature engineering",
            "Reduce regularization strength",
            "Train longer (more epochs)",
        ]
    elif gap > threshold_gap:
        diagnosis = "HIGH VARIANCE (Overfitting)"
        actions = [
            "Get more training data",
            "Increase regularization (L1/L2, dropout)",
            "Reduce model complexity",
            "Use early stopping",
            "Try data augmentation",
        ]
    elif val_score >= 0.9:
        diagnosis = "GOOD FIT"
        actions = [
            "Check calibration (are probabilities reliable?)",
            "Check performance across subgroups",
            "Consider deployment readiness",
        ]
    else:
        diagnosis = "MODERATE FIT (room for improvement)"
        actions = [
            "Try different model architectures",
            "Feature engineering",
            "Hyperparameter tuning",
            "Check data quality",
        ]
    
    print(f"DIAGNOSIS: {diagnosis}")
    print("\nRecommended actions:")
    for i, action in enumerate(actions, 1):
        print(f"  {i}. {action}")
    print("=" * 50)
    return diagnosis

# Examples
print("Example 1: Underfitting")
diagnose_model(0.60, 0.55)
print()
print("Example 2: Overfitting")
diagnose_model(0.99, 0.72)
print()
print("Example 3: Good fit")
diagnose_model(0.95, 0.93)

> **Expected output:** Three diagnostic reports printed, each with train/val scores, gap, diagnosis label (HIGH BIAS, HIGH VARIANCE, or GOOD FIT), and numbered recommended actions.

---
# Section 6: Exercises

## Exercise 1: Debug the Buggy Code

The code below has a **data leakage bug**. The model gets 99% accuracy on the test set, which is suspiciously high. Find the bug and fix it.

In [ ]:
# EXERCISE 1: Find and fix the data leakage bug
# This code has a subtle but critical error

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

np.random.seed(42)
X_ex = np.random.randn(200, 5)
y_ex = (X_ex[:, 0] > 0).astype(int)

# BUG: Scaling BEFORE splitting -- the scaler sees test data statistics!
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_ex)  # <-- LEAKAGE: fit on ALL data including test

X_train_ex, X_test_ex, y_train_ex, y_test_ex = train_test_split(
    X_scaled, y_ex, test_size=0.2, random_state=42)

model_ex = LogisticRegression().fit(X_train_ex, y_train_ex)
print(f"Test accuracy: {model_ex.score(X_test_ex, y_test_ex):.3f}")

# YOUR FIX: Uncomment and complete the code below
# -------------------------------------------------
# X_train_fix, X_test_fix, y_train_fix, y_test_fix = train_test_split(
#     X_ex, y_ex, test_size=0.2, random_state=42)
# scaler_fix = StandardScaler()
# X_train_fix = scaler_fix.fit_transform(______)   # fit on train only
# X_test_fix = scaler_fix.transform(______)         # transform test (NO fit!)
# model_fix = LogisticRegression().fit(X_train_fix, y_train_fix)
# print(f"Fixed test accuracy: {model_fix.score(X_test_fix, y_test_fix):.3f}")

## Exercise 2: Fill in the Learning Curve Analysis

Complete the code to generate a learning curve and diagnose the model.

In [ ]:
# EXERCISE 2: Fill in the blanks to generate and interpret a learning curve

from sklearn.tree import DecisionTreeClassifier

np.random.seed(42)
X_ex2 = np.random.randn(400, 3)
y_ex2 = (X_ex2[:, 0] ** 2 + X_ex2[:, 1] > 0.5).astype(int)

# Fill in: create a decision tree with max_depth=2 (intentionally too shallow)
# shallow_tree = DecisionTreeClassifier(max_depth=______)

# Fill in: generate learning curve
# sizes, train_scores, val_scores = learning_curve(
#     ______, X_ex2, y_ex2,
#     train_sizes=np.linspace(0.1, 1.0, 10), cv=5, scoring='accuracy')

# Fill in: plot the learning curve
# plt.figure(figsize=(8, 5))
# plt.plot(sizes, train_scores.mean(1), 'b-o', label='______')
# plt.plot(sizes, val_scores.mean(1), 'r-o', label='______')
# plt.xlabel('Training Set Size')
# plt.ylabel('Accuracy')
# plt.title('Learning Curve: Shallow Decision Tree')
# plt.legend()
# plt.grid(alpha=0.3)
# plt.show()

# Fill in: what is the diagnosis?
# diagnosis = "______"  # "high bias" or "high variance"?
# print(f"Diagnosis: {diagnosis}")

## Exercise 3: Fill in the Calibration Check

Complete the code to evaluate whether an SVM needs calibration.

In [ ]:
# EXERCISE 3: Check if SVM needs calibration and apply Platt scaling

np.random.seed(42)
X_ex3 = np.random.randn(500, 4)
y_ex3 = (X_ex3[:, 0] + 0.5 * X_ex3[:, 1] > 0).astype(int)
X_train3, X_test3 = X_ex3[:400], X_ex3[400:]
y_train3, y_test3 = y_ex3[:400], y_ex3[400:]

# Step 1: Fit an SVM with probability=True (uses Platt scaling internally)
# svm_uncal = SVC(kernel='rbf', probability=True)
# svm_uncal.fit(______, ______)

# Step 2: Get predicted probabilities
# probs_svm = svm_uncal.predict_proba(______)[:, 1]

# Step 3: Plot calibration curve
# frac, mean_pred = calibration_curve(______, ______, n_bins=10)
# plt.figure(figsize=(6, 6))
# plt.plot([0, 1], [0, 1], 'k--', label='Perfect')
# plt.plot(mean_pred, frac, 'rs-', label='SVM')
# plt.xlabel('Mean Predicted Probability')
# plt.ylabel('Fraction of Positives')
# plt.title('SVM Calibration')
# plt.legend(); plt.grid(alpha=0.3)
# plt.show()

# Step 4: Compute Brier score
# brier = brier_score_loss(______, ______)
# print(f"Brier score: {brier:.4f}")

<details><summary>Exercise Solutions (click to reveal)</summary>

**Exercise 1:** The bug is `scaler.fit_transform(X_ex)` before splitting -- the scaler learns mean/std from ALL data including test. Fix: split first, then `fit_transform` on train, `transform` on test.
```python
X_train_fix = scaler_fix.fit_transform(X_train_fix)
X_test_fix = scaler_fix.transform(X_test_fix)
```

**Exercise 2:**
```python
shallow_tree = DecisionTreeClassifier(max_depth=2)
sizes, train_scores, val_scores = learning_curve(
    shallow_tree, X_ex2, y_ex2, ...)
# Labels: 'Training', 'Validation'
# Diagnosis: "high bias" (both curves converge at moderate accuracy)
```

**Exercise 3:**
```python
svm_uncal.fit(X_train3, y_train3)
probs_svm = svm_uncal.predict_proba(X_test3)[:, 1]
frac, mean_pred = calibration_curve(y_test3, probs_svm, n_bins=10)
brier = brier_score_loss(y_test3, probs_svm)
```

</details>

> **Common Mistake:** In Exercise 1, many students think the fix is to use a Pipeline. While Pipelines do prevent this specific leakage, the deeper lesson is: **never let preprocessing see test data**. Any `fit` or `fit_transform` must happen on training data ONLY. Then use `transform` (not `fit_transform`) on test data.

---
# Section 7: Putting It All Together

A complete diagnostic workflow on a single dataset.

In [ ]:
# Cell 13: Full Diagnostic Workflow

from sklearn.datasets import make_classification

np.random.seed(42)
X_full, y_full = make_classification(
    n_samples=500, n_features=10, n_informative=5,
    n_redundant=2, n_classes=2, random_state=42)

print("STEP 1: Learning Curve")
print("=" * 40)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

models_diag = [
    (LogisticRegression(), 'Logistic Regression'),
    (SVC(kernel='rbf', C=1, probability=True), 'SVM (C=1)'),
    (SVC(kernel='rbf', C=1000, probability=True), 'SVM (C=1000)'),
]

for ax, (model, name) in zip(axes, models_diag):
    sizes, train_sc, val_sc = learning_curve(
        model, X_full, y_full,
        train_sizes=np.linspace(0.1, 1.0, 8), cv=5, scoring='accuracy')
    ax.plot(sizes, train_sc.mean(1), 'b-o', label='Train', markersize=4)
    ax.plot(sizes, val_sc.mean(1), 'r-o', label='Val', markersize=4)
    ax.fill_between(sizes, train_sc.mean(1)-train_sc.std(1),
                    train_sc.mean(1)+train_sc.std(1), alpha=0.1, color='blue')
    ax.fill_between(sizes, val_sc.mean(1)-val_sc.std(1),
                    val_sc.mean(1)+val_sc.std(1), alpha=0.1, color='red')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Training Size')
    ax.set_ylabel('Accuracy')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    gap = train_sc.mean(1)[-1] - val_sc.mean(1)[-1]
    ax.annotate(f'Gap={gap:.3f}', xy=(0.5, 0.02), xycoords='axes fraction',
                fontsize=9, ha='center', color='darkred')
plt.suptitle('Step 1: Learning Curves for Three Models', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 14: Step 2 -- Validation Curve for the best model

print("\nSTEP 2: Validation Curve for SVM C parameter")
print("=" * 40)

C_vals = np.logspace(-2, 4, 15)
train_vc, val_vc = validation_curve(
    SVC(kernel='rbf'), X_full, y_full,
    param_name='C', param_range=C_vals,
    cv=5, scoring='accuracy')

plt.figure(figsize=(8, 5))
plt.semilogx(C_vals, train_vc.mean(1), 'b-o', label='Training', markersize=4)
plt.semilogx(C_vals, val_vc.mean(1), 'r-o', label='Validation', markersize=4)
plt.fill_between(C_vals, train_vc.mean(1)-train_vc.std(1),
                 train_vc.mean(1)+train_vc.std(1), alpha=0.1, color='blue')
plt.fill_between(C_vals, val_vc.mean(1)-val_vc.std(1),
                 val_vc.mean(1)+val_vc.std(1), alpha=0.1, color='red')
plt.xlabel('C')
plt.ylabel('Accuracy')
plt.title('Step 2: Validation Curve for SVM C', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_C = C_vals[np.argmax(val_vc.mean(1))]
print(f"Best C: {best_C:.2f}")
print(f"Best validation accuracy: {val_vc.mean(1).max():.3f}")

In [ ]:
# Cell 15: Step 3 -- Calibration check

print("\nSTEP 3: Calibration Check")
print("=" * 40)

from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X_full, y_full, test_size=0.3, random_state=42)

svm_best = SVC(kernel='rbf', C=best_C, probability=True)
svm_best.fit(X_tr, y_tr)
probs_best = svm_best.predict_proba(X_te)[:, 1]

frac_best, mean_pred_best = calibration_curve(y_te, probs_best, n_bins=10)
brier_best = brier_score_loss(y_te, probs_best)

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], 'k--', label='Perfect')
plt.plot(mean_pred_best, frac_best, 'rs-', label=f'SVM C={best_C:.1f} (Brier={brier_best:.3f})')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.title('Step 3: Calibration of Best Model', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Brier Score: {brier_best:.4f}")
print(f"Test Accuracy: {svm_best.score(X_te, y_te):.3f}")
print("\nDiagnostic workflow complete!")

> **Expected output:** Three-step diagnostic workflow:
> 1. Learning curves for three models showing different bias/variance profiles
> 2. Validation curve identifying optimal C
> 3. Calibration check for the best model
>
> This is the workflow you should follow every time you train a model.

---
# Section 8: Common Errors & Troubleshooting

| Error | Cause | Fix |
|-------|-------|-----|
| `ValueError: Input contains NaN` in sklearn | Missing values in feature matrix | Impute with `SimpleImputer` or drop rows with `dropna()` before fitting |
| Learning curve shows train and val both high loss | Underfitting (high bias) | Increase model complexity, add features, reduce regularization |
| Learning curve shows train low, val high loss | Overfitting (high variance) | More data, increase regularization, reduce model complexity, use dropout |
| Calibration curve shows overconfident predictions | Model outputs are not well-calibrated | Apply Platt scaling (`CalibratedClassifierCV`) or temperature scaling |
| `ConvergenceWarning` from sklearn | Optimizer didn't converge | Increase `max_iter`, scale features with `StandardScaler`, reduce learning rate |
| Validation curve flat across all hyperparameters | Hyperparameter has no effect | Check that the hyperparameter is actually being varied; try a different one |

---
# Section 9: Interview Questions

**Q1 (Easy):** What is the bias-variance tradeoff?
<details><summary>Answer</summary>
Bias is error from wrong assumptions (underfitting) -- the model is too simple to capture the true pattern. Variance is error from sensitivity to training data fluctuations (overfitting) -- the model memorises noise. Total error = Bias^2 + Variance + Irreducible noise. Simple models have high bias, low variance. Complex models have low bias, high variance. The goal is to find the sweet spot that minimises total error, typically through cross-validation.
</details>

**Q2 (Easy):** How do you read a learning curve, and what does it tell you about your model?
<details><summary>Answer</summary>
Plot training and validation error vs. training set size. If both curves are high and converged: underfitting (more data won't help, need a more complex model). If training error is low but validation error is high: overfitting (more data will help, or reduce complexity). If both curves converge to a low value: good fit. The gap between curves indicates variance; the final level indicates bias.
</details>

**Q3 (Medium):** What is model calibration and why does it matter for medical/biological predictions?
<details><summary>Answer</summary>
A calibrated model's predicted probability matches the true frequency -- if it says "70% probability of disease," then 70% of such predictions should actually be positive. Many models (random forests, SVMs, neural nets) are poorly calibrated out of the box. In medicine/biology, miscalibrated probabilities lead to wrong clinical decisions. Fix with Platt scaling (logistic regression on outputs), isotonic regression, or temperature scaling (for neural nets). Evaluate with reliability diagrams and Expected Calibration Error (ECE).
</details>

**Q4 (Medium):** You have a model with 95% accuracy but your stakeholder is unhappy. What metrics should you look at instead?
<details><summary>Answer</summary>
Check: (1) Class distribution -- if 95% of samples are negative, the model could just predict "negative" always. (2) Precision and recall per class -- accuracy hides poor performance on the minority class. (3) F1-score or MCC for imbalanced data. (4) ROC-AUC and PR-AUC (Precision-Recall AUC is better for imbalanced data). (5) Confusion matrix to see exactly where errors occur. (6) Business-relevant metrics -- e.g., cost of false negatives vs. false positives. Always report multiple metrics and choose based on the cost structure of the problem.
</details>

**Q5 (Hard):** Design a diagnostic workflow for a neural network that plateaus at mediocre validation performance. What would you check, in what order?
<details><summary>Answer</summary>
Systematic workflow: (1) Data: Check for label noise, class imbalance, data leakage, insufficient preprocessing. (2) Learning curves: Plot train/val loss over epochs -- is it underfitting (both high) or overfitting (gap)? (3) Gradient diagnostics: Check for vanishing/exploding gradients (plot gradient norms per layer). (4) Architecture: Is the model capacity sufficient? Try larger model to see if training loss can go lower. (5) Hyperparameters: Learning rate (try LR finder), batch size, weight decay. (6) Features: Are inputs properly normalised? Feature importance analysis. (7) Regularization: If overfitting, add dropout, weight decay, data augmentation. (8) Optimiser: Switch Adam to SGD+momentum or vice versa. (9) Ensemble: Combine multiple models. Log everything in a systematic table; change one thing at a time.
</details>

---
# Section 10: Further Reading

**Core references:**

1. Hastie, Tibshirani & Friedman (2009). *The Elements of Statistical Learning*, Ch. 7: Model Assessment and Selection. Springer. DOI: [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7)

2. Niculescu-Mizil & Caruana (2005). *Predicting Good Probabilities with Supervised Learning*. ICML. DOI: [10.1145/1102351.1102430](https://doi.org/10.1145/1102351.1102430)

3. Guo, Pleiss, Sun & Weinberger (2017). *On Calibration of Modern Neural Networks*. ICML. [arXiv:1706.04599](https://arxiv.org/abs/1706.04599)

4. Platt (1999). *Probabilistic Outputs for Support Vector Machines*. Advances in Large Margin Classifiers, MIT Press.

5. Raschka (2018). *Model Evaluation, Model Selection, and Algorithm Selection in Machine Learning*. [arXiv:1811.12808](https://arxiv.org/abs/1811.12808)

**Online resources:**
- [scikit-learn: Model Evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html)
- [scikit-learn: Calibration](https://scikit-learn.org/stable/modules/calibration.html)
- [StatQuest: Bias-Variance Tradeoff](https://www.youtube.com/watch?v=EuBBz3bI-aA)
- [Dive into Deep Learning, Ch. 4.4](https://d2l.ai/chapter_multilayer-perceptrons/underfit-overfit.html)

---
## Notebook Complete

**You can now:**
- [x] Visualize and explain the bias-variance tradeoff with polynomial regression
- [x] Generate and interpret learning curves to diagnose underfitting vs overfitting
- [x] Generate and interpret validation curves to select hyperparameters
- [x] Evaluate model calibration using reliability diagrams and Brier scores
- [x] Apply Platt scaling and temperature scaling to improve calibration
- [x] Follow a systematic diagnostic checklist when a model underperforms
- [x] Debug data leakage bugs in ML pipelines
- [x] Answer 5 common interview questions on model diagnostics

**Next:** `10_openfold3_finetuning/00_openfold3_walkthrough.ipynb` (apply these diagnostics to a real fine-tuning task)